In [66]:
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
import numpy as np
from mplsoccer import Pitch, VerticalPitch
from plottable import ColumnDefinition, Table
from plottable.cmap import normed_cmap
from matplotlib.colors import PowerNorm
from matplotlib.lines import Line2D
from matplotlib.lines import Line2D
from mplsoccer import VerticalPitch
import matplotlib.patches as mpatches
import math

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from great_tables import GT

from extract_carries import extract_carries
import xg_model_baked

from scipy.stats import zscore
from scipy.stats import norm


*Create metrics class*
        
        *Subtask: simple xG model*
        *Subtask: extrapolate carries*

Create Z-Score Creation Class

Create Export Table Class

Forwards
    npXg
    xG assisted (cba figuring out xA fully, will never have to do that in the real world)
    *shots*
    *npGoals*
    *Aerial %*
    *EPV passing*
    *Dribble/Takeon rate*
    *Carrying*
    *Touches in final third*
    Through balls received
    *Through balls played*
    *Fouls won*

Midfielders
    *Progressive pass distance*
    *Progressive passes*
    *Progressive carries*
    *Passes into penalty area*
    *Total tackles*
    *Successful tackles*
    *Tackle %*
    *Pass completion %*
    *Forward passes*
    *Forward pass %*
    *Interceptions (tackles and interceptions?)*
    *Touches*


Defenders
    *Blocked shots*
    *Recoveries*
    *Blocked Crosses*
    *Clearances*

In [67]:
class Metrics:
    def __init__(self, file_path: str, conditions: str = None):
        """
        file_path is path to DuckDB database. conditions is a string for the
        WHERE clause if a specific range of events is needed.
        """
        con = duckdb.connect(file_path)
        if conditions is None:
            self.events = con.sql("select * from events").df()
        else:
            self.events = con.sql(f"select * from events where {conditions}").df()
        self.events['playerId'] = self.events['playerId'].fillna(-1).astype('int64')  # keep start/end slides for through-ball math, give them an int id
        self.players = con.sql("select * from players").df()
        con.close()

        self.events = xg_model_baked.add_xg(self.events)

    def create_metrics(self):
        ev = self.events
        is_shot = ev["type"].isin(["Goal", "SavedShot", "MissedShots", "ShotOnPost"])
        pens = ev[["penaltyScored", "penaltyMissed"]].fillna(False).astype(bool).any(axis=1)

        shots = ev[is_shot]
        shooting_agg = (
            shots.assign(np_shot=(~pens[is_shot]).astype(int),
                         np_goal=((shots["type"] == "Goal") & ~pens[is_shot]).astype(int))
            .groupby("playerId")
            .agg(
                shots=("np_shot", "sum"),
                np_goals=("np_goal", "sum"),
                npxg=("xG", "sum"),
                big_chances_missed=("bigChanceMissed", "sum"),
                big_chances_scored=("bigChanceScored", "sum"),
            )
        )
        shooting_agg["big_chances"] = (
            shooting_agg["big_chances_missed"] + shooting_agg["big_chances_scored"]
        )

        dribbling_agg = (
            ev.groupby("playerId")
            .agg(
                dribbles_lost=("dribbleLost", "sum"),
                dribbles_won=("dribbleWon", "sum"),
            )
        )
        dribbling_agg["dribbles_total"] = (
            dribbling_agg["dribbles_lost"] + dribbling_agg["dribbles_won"]
        )
        dribbling_agg["dribble_success"] = (
            dribbling_agg["dribbles_won"] / dribbling_agg["dribbles_total"].replace(0, np.nan)
        ) * 100

        passing_df = ev[ev["type"] == "Pass"].copy()
        keypass_cols = ["keyPassLong", "keyPassShort", "keyPassCross",
                        "keyPassThroughball", "keyPassOther"]
        passing_df["isKeyPassOP"] = passing_df[keypass_cols].fillna(False).any(axis=1)
        passing_df["isCompleted"] = passing_df["outcomeType"] == "Successful"

        passing_agg_basic = (
            passing_df.groupby("playerId")
            .agg(
                key_passes_open_play=("isKeyPassOP", "sum"),
                through_balls_completed=("passThroughBallAccurate", "sum"),
                passes=("isTouch", "sum"),
                completed_passes=("isCompleted", "sum"),
                pass_epv=("EPV", "sum"),
                forward_passes=("passForward", "sum"),
                big_chances_created=("bigChanceCreated", "sum"),
            )
        )
        passing_agg_basic["completion_percentage"] = (
            passing_agg_basic["completed_passes"]
            / passing_agg_basic["passes"].replace(0, np.nan)
        ) * 100
        passing_agg_basic["forward_pct"] = (
            passing_agg_basic["forward_passes"]
            / passing_agg_basic["passes"].replace(0, np.nan)
        ) * 100

        xga_agg = (
            ev[ev["assist_playerId"].notna()]
            .groupby("assist_playerId")["xG_assisted"]
            .sum()
            .rename("xg_assisted")
        )
        xga_agg.index = xga_agg.index.astype("int64")
        xga_agg.index.name = "playerId"

        adv = ev[(ev["type"] == "Pass") & (ev["outcomeType"] == "Successful")].copy()
        adv["dist_start"] = np.sqrt((100 - adv["x"]) ** 2 + (50 - adv["y"]) ** 2)
        adv["dist_end"] = np.sqrt((100 - adv["endX"]) ** 2 + (50 - adv["endY"]) ** 2)
        adv["progressive_passing_distance"] = (adv["endX"] - adv["x"]).clip(lower=0.0)
        adv["isProgressivePass"] = (adv["x"] > 33.333333) & (
            adv["dist_start"] <= 0.75 * adv["dist_end"]
        )
        adv["penAreaEnd"] = (adv["endX"] >= 83) & (adv["endY"] >= 21) & (adv["endY"] <= 79)

        passing_agg_advanced = (
            adv.groupby("playerId")
            .agg(
                progressive_passing_distance=("progressive_passing_distance", "sum"),
                progressive_passes=("isProgressivePass", "sum"),
                passes_into_pen_area=("penAreaEnd", "sum"),
            )
        )

        defense_agg = (
            ev.groupby("playerId")
            .agg(
                interceptions=("interceptionWon", "sum"),
                tackles_won=("tackleWon", "sum"),
                tackles_lost=("tackleLost", "sum"),
                errors_leading_to_goal=("errorLeadsToGoal", "sum"),
                errors_leading_to_shot=("errorLeadsToShot", "sum"),
                clearances=("clearanceEffective", "sum"),
                recoveries=("ballRecovery", "sum"),
                blocks=("outfielderBlock", "sum"),
                blocked_crosses=("passCrossBlockedDefensive", "sum"),
                aerial_wins=("duelAerialWon", "sum"),
                aerial_losses=("duelAerialLost", "sum"),
                interceptions_in_box=("interceptionIntheBox", "sum"),
            )
        )
        defense_agg["tackles_total"] = defense_agg["tackles_won"] + defense_agg["tackles_lost"]
        defense_agg["tackle_success"] = (
            defense_agg["tackles_won"] / defense_agg["tackles_total"].replace(0, np.nan)
        ) * 100
        defense_agg["aerials"] = defense_agg["aerial_wins"] + defense_agg["aerial_losses"]
        defense_agg["aerial_success"] = (
            defense_agg["aerial_wins"] / defense_agg["aerials"].replace(0, np.nan)
        ) * 100

        touches = ev[ev["isTouch"] == True]
        touches_agg = (
            touches.groupby("playerId")
            .agg(
                touches=("playerId", "count"),
                touches_in_final_third=("finalThird", "sum"),
                fouls_won=("foulGiven", "sum"),
            )
        )

        carries = extract_carries(ev, 2.0)
        carries["progressive_carrying_distance"] = (
            carries["endX"] - carries["startX"]
        ).clip(lower=0.0)
        carries["isProgressive"] = (
            ((carries["startX"] < 60) & (carries["endX"] >= 10.5 + carries["startX"]))
            | ((carries["startX"] > 60) & (carries["endX"] >= 5.25 + carries["startX"]))
            | ((carries["endX"] >= 83) & (carries["endY"] >= 21) & (carries["endY"] <= 79))
        )
        carries["intoPenArea"] = (
            (carries["endX"] >= 83) & (carries["endY"] >= 21) & (carries["endY"] <= 79)
        )
        carries_agg = (
            carries.groupby("playerId")
            .agg(
                progressive_carrying_distance=("progressive_carrying_distance", "sum"),
                progressive_carries=("isProgressive", "sum"),
                carries_into_penalty_area=("intoPenArea", "sum"),
            )
        )

        player_info = self.players[['playerId', 'playerName', 'teamName', 'minutes', 'position']]
        player_info = player_info.set_index('playerId')

        metrics = pd.concat(
            [player_info, shooting_agg, dribbling_agg, passing_agg_basic,
             passing_agg_advanced, defense_agg, touches_agg, carries_agg, xga_agg],
            axis=1,
        )

        metrics = metrics.drop(index=[-1, 0], errors='ignore').fillna(0.0)

        return metrics
    
    def generate_normalized_metrics(self):
        """
        normalizes applicable metrics to per 90 (figuring out TIP/Otip soon)
        """
        df = self.create_metrics().copy()

        
        df['minutes'] = df['minutes'].astype(str).str.replace(',', '').astype(float)

        exclude = {
        'playerName', 'teamName', 'minutes',
        'completion_percentage', 'forward_pct', 'dribble_success',
        'tackle_success', 'aerial_success',
            }
        cols = [c for c in df.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

        nineties = df['minutes'] / 90
        df[cols] = df[cols].div(nineties, axis=0)
        
        return df

    def generate_ratings(self, df_data, position: str, metric_names_and_weights: dict, inclusive: bool = False, minutes_cutoff: int = 200):
        """
        Data input should be normalized

        Metric names and weights should be structured as a dictionary in the below format
        {npxg:30, shots:30}...

        Possible positions: 'FW','MF','DF','GK'
        """

        total = sum(metric_names_and_weights.values())
        if total != 100:
            raise ValueError(f'values must add up to 100. Your values add to {total}')

        demographic_cols = ['playerName', 'teamName', 'position', 'minutes']
        stat_cols = list(metric_names_and_weights.keys())

        cols = demographic_cols + stat_cols

        df = df_data.copy()

        if inclusive == False:
            df = df[df['position'].fillna('').str.contains(position)]
        else:
            df = df[df['position'] == position]

        df = df[df['minutes'] >= minutes_cutoff]

        df = df[cols]
        df_zscores = df.copy()

        for col in stat_cols:
            df_zscores[f'{col}_zscore'] = zscore(df_zscores[col], nan_policy='omit')

        weights = pd.Series(metric_names_and_weights)

        zscore_cols = [f'{metric}_zscore' for metric in weights.index]

        df_zscores['rating'] = (
            df_zscores[zscore_cols]
            .mul(weights.values, axis=1)
            .sum(axis=1)
            / weights.sum()
        )

        df_zscores['rating_100'] = (
            norm.cdf(df_zscores['rating']) * 100
        )

        return df_zscores

In [69]:
obj = Metrics('/Users/owner/Desktop/Natabase/championship_2526.duckdb')
df = obj.generate_normalized_metrics()




/Users/owner/Desktop/GitHub/NextStepsFootballAnallytics/CompRatingArticle/MLSCompRatingArticle/xg_model_baked.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prev = ev.groupby("matchId")["keyPassThroughball"].shift(1).fillna(False).astype(int)


In [78]:
df1 = obj.generate_ratings(position='MF', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 5,'xg_assisted':10, 'tackles_won': 5, 'tackle_success': 5, 'interceptions': 5, 'progressive_passing_distance': 15, 'progressive_passes' :15, 'progressive_carrying_distance': 5, 'passes_into_pen_area': 10, 'aerial_success': 5, 'pass_epv': 15, 'touches': 5}).sort_values('rating_100', ascending = False)

In [79]:
df1

,playerName,teamName,position,minutes,npxg,xg_assisted,tackles_won,tackle_success,interceptions,progressive_passing_distance,...,interceptions_zscore,progressive_passing_distance_zscore,progressive_passes_zscore,progressive_carrying_distance_zscore,passes_into_pen_area_zscore,aerial_success_zscore,pass_epv_zscore,touches_zscore,rating,rating_100
playerId,,,,,,,,,,,,,,,,,,,,,
362246,Alfie Doughty,Millwall,"DF,MF",1401.0,0.040433,0.149603,1.092077,65.384615,1.092077,262.695931,...,0.737504,1.703419,-0.140544,-0.647858,4.691913,1.681182,4.442165,1.726688,1.673217,95.285775
138584,Matt Grimes,Coventry City,MF,4123.0,0.047065,0.083646,1.113267,62.962963,0.742178,416.595440,...,-0.041580,3.733312,1.757206,-0.020141,1.569074,0.617466,3.026494,2.857829,1.589253,94.399832
332402,Ryan Manning,Southampton,"DF,MF",3217.0,0.079936,0.152974,1.063102,62.295082,1.286913,307.440783,...,1.171326,2.293591,-0.794157,1.751132,2.201523,1.024567,3.444383,2.305124,1.412047,92.103198
332143,Gustavo Hamer,Sheffield United,MF,2500.0,0.173688,0.193482,0.684000,59.375000,0.648000,221.616000,...,-0.251277,1.161585,0.041446,0.921266,4.033946,0.210364,3.490130,0.728210,1.402449,91.960933
367217,Imran Louza,Watford,MF,3574.0,0.178903,0.104048,1.007275,67.796610,1.082820,326.067431,...,0.716893,2.539272,1.660049,0.581755,1.196746,0.023318,2.350337,2.470865,1.390196,91.776532
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366120,Macaulay Langstaff,Millwall,MF,1375.0,0.264680,0.019619,0.916364,63.636364,0.261818,30.488727,...,-1.111149,-1.359332,0.118197,-0.981345,-1.283052,-1.375189,-1.345376,-1.984097,-0.855930,19.601835
497156,Rayan Kolli,QPR,"FW,MF",1054.0,0.283897,0.012445,0.512334,75.000000,0.256167,39.381404,...,-1.123732,-1.242040,-0.846072,-0.840752,-1.138363,-1.188689,-1.272504,-1.493489,-0.917386,17.947011
463991,Tom Cannon,Sheffield United,"FW,MF",1726.0,0.279608,0.031056,0.208575,57.142857,0.052144,20.847045,...,-1.578009,-1.486503,-0.363077,-0.685899,-1.253508,-0.142969,-1.148400,-2.380348,-0.952754,17.035743


In [ ]:
df2_wideattacker = obj.generate_ratings(position='FW', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 15,'xg_assisted':35, 'dribble_success': 10, 'dribbles_total': 20, 'shots': 5, 'pass_epv': 5, 'through_balls_completed': 5, 'progressive_carrying_distance': 5}).sort_values('rating_100', ascending = False)

In [83]:
df2_wideattacker

,playerName,teamName,position,minutes,npxg,xg_assisted,dribble_success,dribbles_total,shots,pass_epv,...,npxg_zscore,xg_assisted_zscore,dribble_success_zscore,dribbles_total_zscore,shots_zscore,pass_epv_zscore,through_balls_completed_zscore,progressive_carrying_distance_zscore,rating,rating_100
playerId,,,,,,,,,,,,,,,,,,,,,
442404,Léo Scienza,Southampton,"MF,FW",2401.0,0.209850,0.210179,45.454545,4.947938,3.223657,0.301693,...,-0.708546,3.738314,0.574015,2.708521,1.502500,5.243339,0.549267,3.094588,2.320718,98.984898
405319,Matěj Jurásek,Norwich City,"MF,FW",492.0,0.125341,0.203652,30.000000,1.829268,1.829268,0.199701,...,-1.269459,3.568523,-0.567661,-0.052229,-0.511235,3.235856,2.332821,3.391953,1.413822,92.129294
403757,Anis Mehmeti,Bristol City / Ipswich Town,"MF,FW",3367.0,0.269509,0.136050,45.299145,3.127413,2.673003,0.173975,...,-0.312570,1.809903,0.562535,1.096932,0.707262,2.729493,1.518730,0.456761,1.132833,87.135773
349671,Nathan Broadhead,Ipswich Town / Wrexham,"FW,MF",1926.0,0.270083,0.132854,43.037975,3.691589,2.289720,0.057542,...,-0.308757,1.726756,0.395496,1.596359,0.153736,0.437767,2.398723,0.306348,1.081701,86.030726
313620,Riley McGree,Middlesbrough,"MF,FW",1483.0,0.275755,0.125629,51.428571,2.124073,2.913014,0.083531,...,-0.271115,1.538827,1.015335,0.208742,1.053879,0.949295,1.315945,3.852013,0.999761,84.128680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497156,Rayan Kolli,QPR,"FW,MF",1054.0,0.283897,0.012445,12.500000,1.366224,2.561670,-0.002937,...,-0.217068,-1.405567,-1.860441,-0.462131,0.546478,-0.752637,-0.689277,-0.634374,-0.879470,18.957338
332544,Mark Harris,Oxford United,FW,1034.0,0.194899,0.016839,30.000000,0.870406,1.827853,0.005170,...,-0.807780,-1.291274,-0.567661,-0.901045,-0.513279,-0.593057,-0.689277,-0.255808,-0.912659,18.071089
362819,Liam Cullen,Swansea City,"MF,FW",1737.0,0.227042,0.027117,14.285714,0.725389,2.176166,0.001839,...,-0.594439,-1.023906,-1.728525,-1.029420,-0.010256,-0.658617,-0.689277,-0.659794,-0.927167,17.692003


In [85]:
df3_striker = obj.generate_ratings(position='FW', df_data= df, inclusive = False, minutes_cutoff= 400, metric_names_and_weights={'npxg': 30,'xg_assisted':5, 'shots': 20, 'np_goals': 20, 'pass_epv': 5, 'aerial_success': 10, 'progressive_carrying_distance': 5, 'carries_into_penalty_area': 5 }).sort_values('rating_100', ascending = False)

In [86]:
df3_striker

,playerName,teamName,position,minutes,npxg,xg_assisted,shots,np_goals,pass_epv,aerial_success,...,npxg_zscore,xg_assisted_zscore,shots_zscore,np_goals_zscore,pass_epv_zscore,aerial_success_zscore,progressive_carrying_distance_zscore,carries_into_penalty_area_zscore,rating,rating_100
playerId,,,,,,,,,,,,,,,,,,,,,
481083,Mo Touré,Norwich City,FW,589.0,0.804349,0.095088,3.056027,1.375212,0.032241,24.137931,...,3.237344,0.744311,1.260415,5.410583,-0.060227,-1.079638,-1.276776,-0.944055,2.120602,98.302234
396483,Ellis Simms,Coventry City,FW,1649.0,0.693967,0.022353,4.147968,0.709521,-0.012073,50.289017,...,2.504704,-1.147824,2.837364,2.118365,-0.932447,1.365780,-0.611415,0.504388,1.769770,96.161728
273229,Cyle Larin,Southampton,FW,854.0,0.710099,0.043444,2.740047,0.843091,-0.007978,39.285714,...,2.611776,-0.599150,0.804085,2.778946,-0.851846,0.336849,0.661011,0.920490,1.540349,93.826236
397143,Ross Stewart,Southampton,FW,981.0,0.497429,0.059785,2.568807,0.733945,0.007679,50.000000,...,1.200215,-0.174065,0.556786,2.239155,-0.543679,1.338754,0.816105,2.302268,1.173160,87.963409
338274,Haji Wright,Coventry City,FW,2575.0,0.611831,0.030171,3.355340,0.524272,-0.000126,42.631579,...,1.959540,-0.944453,1.692673,1.202203,-0.697298,0.649724,-0.514226,0.292700,1.138646,87.257458
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363139,Giorgi Chakvetadze,Watford,"MF,FW",1791.0,0.062790,0.156727,1.055276,0.000000,0.144322,17.391304,...,-1.684635,2.347815,-1.629011,-1.390616,2.145830,-1.710523,0.910236,-0.054987,-1.012924,15.554833
491382,Mads Frøkjær-Jensen,Preston North End,"MF,FW",514.0,0.065702,0.018872,0.875486,0.350195,0.056031,23.529412,...,-1.665306,-1.238381,-1.888659,0.341293,0.408027,-1.136541,-0.582715,-0.944055,-1.040575,14.903634
436581,Sverre Nypan,Middlesbrough,"MF,FW",624.0,0.061090,0.103747,1.009615,0.000000,0.037615,40.000000,...,-1.695915,0.969586,-1.694954,-1.390616,0.045554,0.403642,-0.684064,-0.944055,-1.116173,13.217403
